In [11]:
# Install required packages in a Colab cell
#!pip install -qU langgraph langchain-google-genai pdfplumber faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 99.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [2]:
import os
from google.colab import userdata

# Add it to Colab 'Secrets' first
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
os.environ["PINECONE_API_KEY"] = userdata.get('Pinecone')


In [3]:
g_key = userdata.get('GOOGLE_API_KEY')
p_key = userdata.get('Pinecone')

In [5]:
#!pip install langchain_community

In [8]:
!pip install -qU pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.0/349.0 kB 6.9 MB/s eta 0:00:00


In [ ]:
#!pip install -qU langchain-text-splitters

# Chunking

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path = "/content/NVIDIA-2025-Annual-Report.pdf"

# 2. Load and Split
loader = PyPDFLoader(file_path)
data = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(data)

print(f"Original pages: {len(data)}")
print(f"Prepared {len(chunks)} chunks from the PDF")
print(f"First chunk content: {chunks[0].page_content[:200]}...")

Original pages: 181
Prepared 805 chunks from the PDF
First chunk content: 2025
NVIDIA Corporation
Annual Review
Notice of Annual Meeting 
Proxy Statement 
Form 10-K...


#Vectorization (Embeddings) & Storage

In [12]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

# 5/6/26 Replace ChatOpenAI with Gemini 3 Flash Review
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)

In [ ]:
#!pip install "numpy<2.1"

In [14]:
!pip install -qU langchain-pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.0 which is incompatible.


In [15]:
import time
import os
from langchain_pinecone import PineconeVectorStore

embeddings= GoogleGenerativeAIEmbeddings(model= "gemini-embedding-2-preview")
index_name = "nvidia-data-index"
vectorstore = PineconeVectorStore(index_name= index_name, embedding= embeddings)

# batch_size = 50
# total_chunks = len(chunks)

# for i in range(0, total_chunks, batch_size):
#     batch = chunks[i: i+batch_size]
#     vectorstore.add_documents(documents= batch)
#     current_end = min(i + batch_size, total_chunks)
#     print(f"Indexed {current_end}/ {total_chunks}...")

#     #Respect Google API Quota (Pause every 100 chunks)
#     if current_end % 100 == 0 and current_end < total_chunks:
#         print("Pausing 65s for quota reset...")
#         time.sleep(65)

# print("Process Complete! All data is now safely in Pinecone.")

### free tier limits are 100 requests per minute!!

In [16]:
# Check the collection count
from pinecone import Pinecone
import os

pc = Pinecone(api_key= os.environ.get("PINECONE_API_KEY"))
#Target specific index
index = pc.Index("nvidia-data-index")

#Fetch and print the stats
stats = index.describe_index_stats()
print(f"Total vectors in index: {stats['total_vector_count']}")

Total vectors in index: 805


In [17]:
# Load and test the 805 chunks we already saved
query = "What does the report say about AI chip demand?"
docs = vectorstore.similarity_search(query)
print(docs[0].page_content)

Demand estimates for our products, applications, and services can be incorrect and create volatility in our revenue or 
supply levels. We may not be able to generate significant revenue from them. Advancements in accelerated computing 
and generative AI models, along with the growth in model complexity and scale, have driven increased demand for our 
Data Center systems.
We continue to increase our supply and capacity purchases with existing and new suppliers to support our demand 
projections and increasing complexity of our data center products. With these additions, we have also entered and may 
continue to enter into prepaid manufacturing and capacity agreements to supply both current and future products. The 
increased purchase volumes and integration of new suppliers and contract manufacturers into our supply chain creates 
more complexity in managing multiple suppliers with variations in production planning, execution and logistics. Our


# Define the retreival chain

In [ ]:
#%pip install langchain-google-genai langchain-chroma

In [ ]:
#%pip install -U langchain-classic

In [18]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Define the system prompt
system_prompt = (
    "You are an expert financial analyst. Use the following pieces of retrieved context "
    "from the NVIDIA annual report to answer the user's queestion. If you do you know the answer, "
    "just say you don't know. Keep the ansewr concise and professional."
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)
# 2. Link the LLM and the Vector Store
retriever = vectorstore.as_retriever(search_kwargs={"k":5})
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain= create_retrieval_chain(retriever, question_answer_chain)

print("RAG Chain is ready, You can now ask questions!")

RAG Chain is ready, You can now ask questions!


In [19]:
response = rag_chain.invoke({"input": "Summarized NVIDIA's competitive advantages in the AI data center market according to this report."})

print("### NVIDIA AI Market Analysis:")
print(response["answer"])

### NVIDIA AI Market Analysis:
Based on the report, NVIDIA’s competitive advantages in the AI data center market include:

*   **Full-Stack & End-to-End Infrastructure:** NVIDIA provides a complete AI platform spanning hardware (Blackwell GPUs, Grace CPUs, and BlueField DPUs), networking (NVLink, InfiniBand, and Ethernet), and software. This deep integration ensures world-class performance and differentiation.
*   **Software Ecosystem Moat:** With over 5.9 million developers using CUDA and other tools, NVIDIA benefits from a "virtuous cycle." Its **NVIDIA AI Enterprise** suite—including NIM (Inference Microservices), NeMo, and AI Blueprints—simplifies the development and deployment of production-grade generative AI.
*   **Unified Platform Strategy:** A single, programmable architecture allows NVIDIA to address diverse multi-billion-dollar markets. This architecture is available through virtually every major server manufacturer and Cloud Service Provider (CSP).
*   **Market Leadership i

### Add Sources/ Citations to the response

In [20]:
response = rag_chain.invoke({"input": "What were NVIDIA's key growth drivers for the Data Center business in fiscal year 2025?"})

print("### NVIDIA AI Market Analysis:")
print(response["answer"])
print("--- Sources Retrieved---")
for i, doc in enumerate(response["context"]):
    # Printing the first 150 characters of each source chunk
    print(f"Source {i+1}: {doc.page_content[:150]}...")

### NVIDIA AI Market Analysis:
In fiscal year 2025, NVIDIA’s Data Center revenue grew 142% to $115.2 billion, driven by the following key factors:

*   **Hopper Architecture:** Exceptional demand for accelerated computing platforms used for large language models (LLMs), recommendation engines, and generative AI applications.
*   **Blackwell Architecture:** The launch and production shipment of the Blackwell platform, which contributed $11 billion in revenue in the fourth quarter alone, marking the fastest product ramp in the company’s history.
*   **Networking Solutions:** Significant uptake of "Ethernet for AI," specifically the **Spectrum-X** end-to-end platform, and new growth opportunities from **NVLink**.
*   **Workload Transition:** A shift from AI training toward real-time reasoning and inference, which has become the dominant workload for the segment.
--- Sources Retrieved---
Source 1: demand for our Hopper architecture accelerated computing platform used for large language mod

### Q&A
Why k:5?: I chose to retrieve 5 chunks because financial reports are dense. One chunk might mention a number, but the next chunk might provide the necessary context (like "in millions" or "adjusted for inflation"). k:5 strikes a balance between providing enough context and staying within the LLM's prompt limits.

Why create_stuff_documents_chain?: Since I am using a modern model like Gemini, it has a large "context window." "Stuffing" all 5 chunks into the prompt is the most efficient way to give the model the full picture without losing details through summarization.

### Streamlit

In [59]:
!pip install streamlit -q
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

In [49]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
changed 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [56]:
%%writefile app.py
import os
import streamlit as st
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# 1. API KEY SETUP
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "")
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY", "")

# 2. CACHED INITIALIZATION (The "Speed Hack")
@st.cache_resource
def initialize_rag_chain():
    # Load embeddings once
    embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

    # Load Vectorstore once
    index_name = "nvidia-data-index"
    vectorstore = PineconeVectorStore(
        index_name=index_name,
        embedding=embeddings
    )

    # Initialize LLM once
    llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)

    # Define Chain Logic once
    system_prompt = (
        "You are an expert financial analyst. Use the following pieces of retrieved context "
        "from the NVIDIA annual report to answer the user's question. If you don't know the answer, "
        "just say you don't know. Keep the answer concise and professional.\n\n{context}"
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    return create_retrieval_chain(retriever, question_answer_chain)

# Call the cached function
rag_chain = initialize_rag_chain()

# 3. STREAMLIT UI
st.set_page_config(page_title="NVIDIA Analyst Bot", page_icon="📈")
st.title("NVIDIA Financial Analyst")

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if user_input := st.chat_input("Ask about NVIDIA's FY2025 revenue..."):
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)

    with st.chat_message("assistant"):
        with st.spinner("Searching annual report..."):
            response = rag_chain.invoke({"input": user_input})
            answer = response["answer"]
            st.markdown(answer)
            with st.expander("📚 Source Citations"):
                for doc in response["context"]:
                    st.info(doc.page_content)

    st.session_state.messages.append({"role": "assistant", "content": answer})

Overwriting app.py


In [ ]:
!streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > streamlit.log 2>&1 &
!./cloudflared tunnel --url http://localhost:8501

2026-06-23T04:17:00Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-23T04:17:00Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-23T04:17:04Z INF +--------------------------------------------------------------------------------------------+
2026-06-23T04:17:04Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-23T04:17:04Z INF |  https://developers-measure-offshore-himself.trycloudf

#### Turn Retrieval into a Tool

In [24]:
embeddings = GoogleGenerativeAIEmbeddings(
    model = "gemini-embedding-2-preview"
)

vectorstore = PineconeVectorStore(
    index_name = "nvidia-data-index",
    embedding= embeddings,
)

retriever= vectorstore.as_retriever(search_kwards={"k":5})

In [25]:
from langchain_core.tools import tool
@tool
def search_nvidia_annual_report(question: str) -> str:
    """Search the Nvidia FY2025 annual report for financial facts, risks, revenue, segments, and business details."""
    docs= retriever.invoke(question)
    return "\n\n".join([doc.page_content for doc in docs])

In [26]:
result = search_nvidia_annual_report.invoke(
    "What was NVIDIA's revenue in FY2025?"
)
print(result)

BUSINESS OVERVIEW
Fiscal 2025 marked an extraordinary year for NVIDIA’s growth with revenue surging 114% year on year to $130.5 billion on 
strength across all market platforms. Growth was led by exceptional Data Center demand for our Hopper architecture 
used for large language models, recommendation engines, and generative AI applications.  Ethernet for AI was another 
key contributor, including strong uptake of our Spectrum-X end-to-end ethernet platform.  Gross margin expanded year 
on year to 75.0% and we drove strong operating leverage with operating income rising 147% to $81.5 billion and diluted 
earnings per share increasing 147% to $2.94.
Fiscal 2025 Results
Revenue Gross Margin Operating Income Diluted Earnings Per 
Share
$130.5 billion 75.0% $81.5 billion $2.94
up 114% year on year  up 2.3 points year on year up 147% year on year up 147% year on year 
Fiscal 2025 Reportable Segments
Our two reportable segments are “Compute & Networking” and “Graphics”:

Year Ended
  Jan 26,

#### give the LLM access to my NVIDIA search tool.

In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = "gemini-3-flash-preview",
    temperature= 0
)

llm_with_tools = llm.bind_tools([search_nvidia_annual_report])

#### Test the LLM

In [28]:
response = llm_with_tools.invoke(
    "What was NVIDIA's revenue in FY2025?"
)

####Run the tool call

In [29]:
response.tool_calls

[{'name': 'search_nvidia_annual_report',
  'args': {'question': "What was NVIDIA's revenue in FY2025?"},
  'id': 'jveb56h0',
  'type': 'tool_call'}]

In [30]:
tool_call = response.tool_calls[0]

tool_result = search_nvidia_annual_report.invoke(tool_call["args"])

print(tool_result)

BUSINESS OVERVIEW
Fiscal 2025 marked an extraordinary year for NVIDIA’s growth with revenue surging 114% year on year to $130.5 billion on 
strength across all market platforms. Growth was led by exceptional Data Center demand for our Hopper architecture 
used for large language models, recommendation engines, and generative AI applications.  Ethernet for AI was another 
key contributor, including strong uptake of our Spectrum-X end-to-end ethernet platform.  Gross margin expanded year 
on year to 75.0% and we drove strong operating leverage with operating income rising 147% to $81.5 billion and diluted 
earnings per share increasing 147% to $2.94.
Fiscal 2025 Results
Revenue Gross Margin Operating Income Diluted Earnings Per 
Share
$130.5 billion 75.0% $81.5 billion $2.94
up 114% year on year  up 2.3 points year on year up 147% year on year up 147% year on year 
Fiscal 2025 Reportable Segments
Our two reportable segments are “Compute & Networking” and “Graphics”:

Year Ended
  Jan 26,

#### Give the Tool Result Back to the LLM

In [31]:
from langchain_core.messages import HumanMessage, ToolMessage

question = "What was NVIDIA's revenue in FY2025?"

messages = [HumanMessage(content=question)]

response = llm_with_tools.invoke(messages)

#run the tool
tool_call = response.tool_calls[0]

tool_result = search_nvidia_annual_report.invoke(
    tool_call["args"]
)

# Wrap tool result
tool_message = ToolMessage(
    content= tool_result,
    tool_call_id= tool_call["id"]
)

#send the full message back
final_response = llm_with_tools.invoke([
    HumanMessage(content=question),
    response,
    tool_message
])

print(final_response.content)

[{'type': 'text', 'text': "NVIDIA's revenue in fiscal year 2025 was **$130.5 billion** (specifically $130,497 million). This represented a 114% increase from the $60.9 billion reported in fiscal year 2024.", 'extras': {'signature': 'Eq0CCqoCAQw51sfk0SL9VuZ4ZhOVv8N7FxY/KrP7U9FLp2QZBnNa3HS2dU3JNqcQ7ruddAePdWzfA6oj8ffWCcWxFi/+vgkTApGNGB4b06EOqWfLEZ+yX4ijW0DQYsWqsj9whSDJxQ7+aC0CxnsjjLuAqPHrHXUj9nKWaMpC5AeWGpnqRtjGNcbPZAFJay3rSJDTsXH0fB4QrqZb6f5LYI/X+O0ExTXQzbXL2/uPdn2AZiU1yldO+kxT0svRVx6V8TxqHPsaJ4p9J/I3Hr0Fhh0nupgVx6uT6LvqUvKdwJsPXIy/CE7a6smtTflgPLQZ6osK0KigAeT0A5eXDPG0+vBGyXMOrkdWw5c5q5JMNszcc4lkzWqttUEZII3al/oUfHrikkQ+ki+1ocI+o3Rh7Q=='}}]


#### Reasoning and Acting (ReAct)

In [32]:
from langgraph.prebuilt import create_react_agent

tools = [search_nvidia_annual_report]

# 2. Define the System Prompt
system_message = (
    "You are an expert financial analyst. "
    "Use the NVIDIA annual report tool when needed to provide accurate, "
    "data-driven insights."
)

agent_executor = create_react_agent(
    model=llm,
    tools = tools,
    prompt= system_message

)

result = agent_executor.invoke({
    "messages": [("human", "What was NVIDIA's revenue growth in FY2025?")]
})

print(result["messages"][-1].content)


/tmp/ipykernel_502/248923916.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


[{'type': 'text', 'text': "NVIDIA's revenue for fiscal year 2025 was **$130.5 billion**, representing a **114% increase** compared to the previous year.\n\nKey drivers of this growth included:\n*   **Data Center Demand:** Revenue in this segment grew 142% year-over-year, fueled by exceptional demand for the Hopper architecture used in large language models and generative AI.\n*   **Compute & Networking Segment:** This segment saw a 145% increase, reaching $116.2 billion.\n*   **Automotive:** Revenue grew 55% due to sales of self-driving platforms.\n*   **Professional Visualization:** Revenue increased 21% driven by the ramp of Ada RTX GPU workstations.\n*   **Gaming:** Revenue grew 9% year-over-year, supported by sales of GeForce RTX 40 Series GPUs."}]


In [ ]:
from langchain_core.messages import ToolMessage

# 1. Update your model node to use the tool-bound version
def call_model(state: AgentState):
    # CRITICAL: Use llm_with_tools so the model can actually decide to call tools!
    response = llm_with_tools.invoke(state['messages'])
    return {"messages": [response]}

# 2. Update your tool node to execute your real Pinecone search tool
def call_tool(state: AgentState):
    """Executes the actual tool requested by the model and returns a ToolMessage."""
    last_message = state['messages'][-1]

    tool_outputs = []
    # Loop through any tool calls requested by the LLM
    for tool_call in last_message.tool_calls:
        if tool_call["name"] == "search_nvidia_annual_report":
            # Invoke your real retriever tool using the arguments given by the model
            content = search_nvidia_annual_report.invoke(tool_call["args"])

            # Wrap the raw output string in an official ToolMessage
            tool_message = ToolMessage(
                content=content,
                tool_call_id=tool_call["id"],
                name=tool_call["name"]
            )
            tool_outputs.append(tool_message)

    return {"messages": tool_outputs}

In [40]:
from langgraph.prebuilt import create_react_agent

app = create_react_agent(model= llm, tools= [search_nvidia_annual_report])

/tmp/ipykernel_502/2714677934.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  app = create_react_agent(model= llm, tools= [search_nvidia_annual_report])


### Test the custom agent

In [41]:
from langchain_core.messages import HumanMessage

inputs = {"messages": [HumanMessage(content= "What was NVIDIA's revenue in FY2025?")]}

# Use stream to watch the state mutate at every node transition
for output in app.stream(inputs):
    for node, state in output.items():
        print(f"\n--- Node Executed: {node} ---")
        if state.get("messages"):
            print(state["messages"][-1])


--- Node Executed: agent ---
content=[] additional_kwargs={'function_call': {'name': 'search_nvidia_annual_report', 'arguments': '{"question": "What was NVIDIA\'s total revenue for fiscal year 2025?"}'}, '__gemini_function_call_thought_signatures__': {'t8rdtsvf': 'ErkBCrYBAQw51sejF6B6103d2sVWcy0RZNQEtlF4inbClv/2CZi3mN9kHPy3wpn0/MVkF+B8yL3j8eCfJNUSOouDZtHPFwCQMlYBWB4RmvqTSthipGqw+fer9vyW9vIEeJ5E+mtWn2d3gY1k2d2GiNG3hgnOcw+Icv69yQ/w5yuuY8oVtoLKtZw0+cHPoHQDjatL5uHSVhSqUOrFo+8w3rwpsEcjX6MpR7Oxe8QHc0/+RchBf3COjgIezQ0='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019ef28a-9a8e-7102-94b4-b7663c98fa99-0' tool_calls=[{'name': 'search_nvidia_annual_report', 'args': {'question': "What was NVIDIA's total revenue for fiscal year 2025?"}, 'id': 't8rdtsvf', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 79, 'output_tokens': 70, 'total_tokens': 149, 'input_token_de